# Lab Assignment: Exploratory Data Analysis with Pandas

**Goal**: Implement data manipulation and visualization using only Pandas-native functions.

**Note**: Use the provided synthetic data to complete the questions. Focus on using `df.plot()` and other Pandas-specific methods.

In [1]:
import pandas as pd
import numpy as np

# Setup for the lab
np.random.seed(42)
n = 200
data = {
    'product_id': range(1, n + 1),
    'category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Groceries'], n),
    'price': np.random.uniform(10, 500, n).round(2),
    'units_sold': np.random.randint(1, 100, n),
    'rating': np.random.uniform(1, 5, n).round(1),
    'in_stock': np.random.choice([True, False], n)
}
df = pd.DataFrame(data)
print('Dataset prepared for assignment.')

Dataset prepared for assignment.


### Question 1: Initial Exploration

**Task**: Use Pandas functions to identify the total number of rows/columns, the data types of each column, and the count of missing values.

**Hints**:
- Check `.shape` for dimensions.
- Use `.info()` for types and null counts.
- Use `.describe()` for a statistical overview.

In [2]:
# Question 1: Initial Exploration

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum())

print("\nStatistical summary:")
print(df.describe(include="all"))

### Question 2: Advanced Filtering

**Task**: Create a new DataFrame containing only the 'Electronics' and 'Clothing' categories where the `price` is above the average price of the entire dataset.

**Hints**:
- Use `.mean()` to calculate the threshold.
- Use boolean indexing or `.query()` for filtering.

In [3]:
# Question 2: Advanced Filtering

avg_price = df["price"].mean()

filtered_df = df[
    (df["category"].isin(["Electronics", "Clothing"]))
    & (df["price"] > avg_price)
].copy()

print("Average price:", round(avg_price, 2))
print("Rows after filtering:", filtered_df.shape[0])
filtered_df.head()

### Question 3: Grouping and Aggregation

**Task**: Find the total revenue (Price * Units Sold) per category. Sort the result from highest to lowest revenue.

**Hints**:
- Create a new column first.
- Use `.groupby()` and `.sum()`.
- Use `.sort_values()`.

In [4]:
# Question 3: Grouping and Aggregation

# Create revenue column first
df["revenue"] = df["price"] * df["units_sold"]

revenue_per_category = (
    df.groupby("category", as_index=False)["revenue"]
      .sum()
      .sort_values("revenue", ascending=False)
)

print(revenue_per_category)

### Question 4: Data Visualization (Pandas Only)

**Task**: Generate the following plots using the `df.plot()` method:
1. A **bar chart** showing the average rating per category.
2. A **scatter plot** relating 'price' to 'units_sold'.
3. A **histogram** of the 'price' distribution.

**Hints**:
- For the bar chart, you must aggregate the data first.
- Use the `kind` parameter (e.g., `kind='bar'`, `kind='scatter'`, `kind='hist'`).

In [5]:
# Question 4: Data Visualization (Pandas Only)

import matplotlib.pyplot as plt

# 1) Bar chart: average rating per category
avg_rating_df = (
    df.groupby("category", as_index=False)["rating"]
      .mean()
      .rename(columns={"rating": "avg_rating"})
)

ax1 = avg_rating_df.plot(
    kind="bar",
    x="category",
    y="avg_rating",
    legend=False,
    title="Average Rating per Category"
)
ax1.set_xlabel("Category")
ax1.set_ylabel("Average Rating")

# 2) Scatter plot: price vs units_sold
ax2 = df.plot(
    kind="scatter",
    x="price",
    y="units_sold",
    alpha=0.7,
    title="Price vs Units Sold"
)
ax2.set_xlabel("Price")
ax2.set_ylabel("Units Sold")

# 3) Histogram: distribution of price
ax3 = df.plot(
    kind="hist",
    y="price",
    bins=20,
    title="Price Distribution"
)
ax3.set_xlabel("Price")

plt.tight_layout()
plt.show()

### Question 5: Pivot Tables

**Task**: Create a pivot table that shows the **average units_sold** for each `category`, broken down by whether the items are `in_stock` or not.

**Hints**:
- Use `df.pivot_table()`.
- Set `index='category'`, `columns='in_stock'`, and `values='units_sold'`.
- Use `aggfunc='mean'`.

In [6]:
# Question 5: Pivot Tables

pivot_table = df.pivot_table(
    index="category",
    columns="in_stock",
    values="units_sold",
    aggfunc="mean"
)

print(pivot_table)

### Question 6: Binning and Custom Mapping

**Task**: Create a new column called `price_bracket`. If the price is < 100, label it 'Budget'; if 100-300, label it 'Mid-Range'; if > 300, label it 'Premium'. Then, count how many products fall into each bracket.

**Hints**:
- Use `pd.cut()` to create the bins.
- Use `.value_counts()` to see the distribution.

In [7]:
# Question 6: Binning and Custom Mapping

bins = [-np.inf, 100, 300, np.inf]
labels = ["Budget", "Mid-Range", "Premium"]

df["price_bracket"] = pd.cut(
    df["price"],
    bins=bins,
    labels=labels,
    right=False,  # Budget: price < 100; Mid-Range: 100 <= price < 300
    include_lowest=True
)

# Per requirement: treat price == 300 as Mid-Range
df.loc[df["price"] == 300, "price_bracket"] = "Mid-Range"

price_bracket_counts = df["price_bracket"].value_counts().reindex(labels)
print(price_bracket_counts)